<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase26_gestion_riesgo_carry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 26: gestión de riesgo del carry

Se toma el carry de la clase 25 y se prueban dos reglas de riesgo (volatilidad objetivo de la cartera y reducción de exposición tras una caída) para ver si reducen las pérdidas grandes sin empeorar el Sharpe.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# EJEMPLO A MANO 1: apalancamiento con volatilidad objetivo
# apalancamiento = volatilidad objetivo / volatilidad estimada (con tope). Objetivo de 4.5% anual, tope de 2 veces.
objetivo, tope = 0.045, 2.0
for vol_est in (0.02, 0.03, 0.045, 0.06, 0.09):
    print("Volatilidad estimada %.1f%% -> apalancamiento %.2f" % (vol_est * 100, min(objetivo / vol_est, tope)))
# Con volatilidad de 2%, objetivo/vol da 2.25 y el tope lo corta en 2.0. Con 9%, reduces a la mitad.

# EJEMPLO A MANO 2: por que importa la caida maxima
# Para recuperar una perdida de x% se necesita ganar x / (1 - x)
for perdida in (0.0743, 0.10, 0.20, 0.50):
    print("Perdida de %.2f%% -> hay que ganar %.2f%% para volver al punto de partida" % (perdida * 100, perdida / (1 - perdida) * 100))

# EJEMPLO A MANO 3: regla de caida (media exposicion si la caida pasa de 3%)
pnl = np.array([0.004, 0.003, -0.02, -0.015, 0.01, 0.012])      # 6 dias de un carry, en %
cum = pnl.cumsum(); caida = cum - np.maximum.accumulate(cum)
exposicion = np.where(caida < -0.03, 0.5, 1.0)                    # se decide al cierre y se aplica al dia siguiente
print("\nAcumulado :", (cum * 100).round(2))
print("Caida      :", (caida * 100).round(2))
print("Exposicion que se usara al dia siguiente:", exposicion)


Volatilidad estimada 2.0% -> apalancamiento 2.00
Volatilidad estimada 3.0% -> apalancamiento 1.50
Volatilidad estimada 4.5% -> apalancamiento 1.00
Volatilidad estimada 6.0% -> apalancamiento 0.75
Volatilidad estimada 9.0% -> apalancamiento 0.50
Perdida de 7.43% -> hay que ganar 8.03% para volver al punto de partida
Perdida de 10.00% -> hay que ganar 11.11% para volver al punto de partida
Perdida de 20.00% -> hay que ganar 25.00% para volver al punto de partida
Perdida de 50.00% -> hay que ganar 100.00% para volver al punto de partida

Acumulado : [ 0.4  0.7 -1.3 -2.8 -1.8 -0.6]
Caida      : [ 0.   0.  -2.  -3.5 -2.5 -1.3]
Exposicion que se usara al dia siguiente: [1.  1.  1.  0.5 1.  1. ]


In [2]:
# Funciones (las mismas de la clase 25, mas medidas de cola)
def vol_ewma(rend, lam=0.94):
    """Volatilidad anualizada con EWMA, usando datos hasta t (inclusive)."""
    return np.sqrt((rend ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)

def backtest(rend, pos, rezago=1, costo_bps=0.0):
    """La posicion al cierre de t gana el rendimiento de t+1 (rezago=1). 'rend' es el rendimiento de estar largo USD."""
    pos_usada = pos.shift(rezago)
    bruto = (pos_usada * rend).mean(axis=1)
    rotacion = pos_usada.diff().abs().mean(axis=1).fillna(0.0)
    return bruto - rotacion * costo_bps / 10000, rotacion

def sharpe(x):
    x = x.dropna()
    return x.mean() / x.std() * np.sqrt(252)

def metricas(pnl):
    pnl = pnl.dropna()
    anios = len(pnl) / 252
    s = sharpe(pnl)
    t = s * np.sqrt(anios)
    return {"rend_%": pnl.mean() * 252 * 100, "vol_%": pnl.std() * np.sqrt(252) * 100, "Sharpe": s,
            "t": t, "p": 2 * (1 - norm.cdf(abs(t)))}

def caida_maxima(pnl):
    c = pnl.dropna().cumsum()
    return (c - c.cummax()).min() * 100


def cola(pnl):
    pnl = pnl.dropna()
    mensual = pnl.resample("ME").sum() * 100
    peores = pnl.sort_values().iloc[: max(1, int(len(pnl) * 0.05))]
    return {"peor_mes_%": mensual.min(), "asimetria": mensual.skew(), "caida_max_%": caida_maxima(pnl),
            "media_peor_5%_dia_%": peores.mean() * 100}


In [3]:
# Datos (igual que la clase 25): precios de seis pares contra el dolar y tasas de interes de corto plazo (FRED, mensuales)
pares = {"USD/MXN": ("MXN=X", False, "MXN"), "USD/JPY": ("JPY=X", False, "JPY"), "USD/EUR": ("EURUSD=X", True, "EUR"),
         "USD/GBP": ("GBPUSD=X", True, "GBP"), "USD/CAD": ("CAD=X", False, "CAD"), "USD/BRL": ("BRL=X", False, "BRL")}
px = {}
for nombre, (ticker, invertir, _) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False, auto_adjust=True)["Close"].squeeze()
    px[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(px).dropna().pct_change().dropna()     # rendimiento de estar largo USD (precio en moneda extranjera por dolar)
CALENTAMIENTO = 505                                        # las mismas fechas de evaluacion de la clase 24
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date())

# Tasas de corto plazo de FRED (una por mes). Para cada moneda se prueban varias series, en este orden, y se usa la primera que baje y este al dia.
ids_tasa = {"USD": ["IR3TIB01USM156N", "IRSTCI01USM156N", "FEDFUNDS"],
            "MXN": ["IR3TIB01MXM156N", "IRSTCI01MXM156N", "INTDSRMXM193N"],
            "JPY": ["IR3TIB01JPM156N", "IRSTCI01JPM156N", "INTDSRJPM193N"],
            "EUR": ["IR3TIB01EZM156N", "IRSTCI01EZM156N", "ECBDFR"],
            "GBP": ["IR3TIB01GBM156N", "IRSTCI01GBM156N", "INTDSRGBM193N"],
            "CAD": ["IR3TIB01CAM156N", "IRSTCI01CAM156N", "INTDSRCAM193N"],
            "BRL": ["IRSTCI01BRM156N", "INTDSRBRM193N", "IR3TIB01BRM156N"]}
def tasa_fred(serie):
    s = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={serie}", index_col=0, parse_dates=True).iloc[:, 0]
    s = pd.to_numeric(s, errors="coerce").dropna()
    return s.resample("MS").mean().dropna()                # una observacion por mes (promedio), tambien para series diarias como ECBDFR
limite = rend.index[-1] - pd.DateOffset(months=6)          # una serie que termina antes de esta fecha se considera desactualizada
tasas, usada = {}, {}
for moneda, lista in ids_tasa.items():
    bajadas = {}
    for sid in lista:
        try:
            bajadas[sid] = tasa_fred(sid)
        except Exception as e:
            print("   no sirvio %s (%s): %s" % (moneda, sid, str(e)[:60]))
    if not bajadas:
        continue
    al_dia = [sid for sid in bajadas if bajadas[sid].index[-1] >= limite]
    elegida = al_dia[0] if al_dia else max(bajadas, key=lambda k: bajadas[k].index[-1])
    tasas[moneda], usada[moneda] = bajadas[elegida], elegida
    if not al_dia:
        print("   AVISO: ninguna serie de %s esta al dia; se usa %s" % (moneda, elegida))
faltan = [m for m in ids_tasa if m not in tasas]
if faltan:
    print("SIN TASA para:", faltan, "-> se quitan esos pares del analisis (ya no son seis, y eso cambia la comparacion con la clase 24)")
    pares = {k: v for k, v in pares.items() if v[2] in tasas}
    rend = rend[list(pares)]
if "USD" not in tasas:
    raise SystemExit("Sin tasa del dolar. Mandame este mensaje.")
for m, s in tasas.items():
    print("%s (%s): ultima fecha %s | ultimo valor %.2f%% | promedio desde 2020-10: %.2f%%" % (m, usada[m], s.index[-1].date(), s.iloc[-1], s.loc["2020-10-01":].mean()))

# Sin mirar el futuro: la tasa del mes m (publicada despues) solo se usa a partir del mes m+1
tasas_df = pd.DataFrame(tasas)
tasas_df.index = tasas_df.index + pd.DateOffset(months=1)
tasas_d = tasas_df.reindex(tasas_df.index.union(rend.index)).ffill().reindex(rend.index)
for m, s in tasas.items():
    if s.index[-1] + pd.DateOffset(months=1) < rend.index[-1]:
        print("AVISO: la tasa de %s se congela desde %s (ultimo dato de FRED)" % (m, s.index[-1].date()))

# Diferencial anual: tasa del dolar menos tasa de la otra moneda (positivo: estar largo USD cobra intereses)
dif = pd.DataFrame({p: (tasas_d["USD"] - tasas_d[c]) / 100 for p, (_, _, c) in pares.items()})
rend_int = dif / 252                                       # intereses diarios de estar largo USD
rend_tot = rend + rend_int                                 # rendimiento total de estar largo USD
print("\nDiferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:")
print((dif.iloc[CALENTAMIENTO:].mean() * 100).round(2).to_string())


Dias: 1570 | del 2020-10-08 al 2026-10-21
   no sirvio BRL (IR3TIB01BRM156N): HTTP Error 404: Not Found
USD (IR3TIB01USM156N): ultima fecha 2026-10-01 | ultimo valor 4.12% | promedio desde 2020-10: 3.28%
MXN (IR3TIB01MXM156N): ultima fecha 2026-10-01 | ultimo valor 6.81% | promedio desde 2020-10: 8.43%
JPY (IR3TIB01JPM156N): ultima fecha 2026-09-01 | ultimo valor 1.68% | promedio desde 2020-10: 0.32%
EUR (ECBDFR): ultima fecha 2026-11-01 | ultimo valor 2.50% | promedio desde 2020-10: 1.74%
GBP (IRSTCI01GBM156N): ultima fecha 2026-10-01 | ultimo valor 3.73% | promedio desde 2020-10: 3.03%
CAD (IR3TIB01CAM156N): ultima fecha 2026-10-01 | ultimo valor 2.45% | promedio desde 2020-10: 2.61%
BRL (IRSTCI01BRM156N): ultima fecha 2026-10-01 | ultimo valor 13.75% | promedio desde 2020-10: 11.22%
AVISO: la tasa de JPY se congela desde 2026-09-01 (ultimo dato de FRED)

Diferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:
USD/MXN   -5.30
USD/JPY    4.05
USD/EUR    1.80
US

In [4]:
# TRES VERSIONES DEL CARRY, todas con rendimiento total (precio mas intereses)
# A: el carry de la clase 25. B: apalancamiento por volatilidad de la cartera. C: media exposicion si la caida pasa de 3%.
escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
carry = np.sign(dif)
pos_A = carry * escala
pnl_A_full = backtest(rend_tot, pos_A)[0]

OBJETIVO, TOPE, UMBRAL = 0.045, 2.0, 0.03                  # fijados antes de ver resultados; no se optimizan
vol_cartera = np.sqrt((pnl_A_full ** 2).ewm(alpha=0.06, adjust=False).mean() * 252)   # EWMA con lambda 0.94, datos hasta t
lev_B = (OBJETIVO / vol_cartera).clip(upper=TOPE).fillna(1.0)
cum = pnl_A_full.fillna(0).cumsum()
lev_C = pd.Series(np.where(cum - cum.cummax() < -UMBRAL, 0.5, 1.0), index=cum.index)

pos_B = pos_A.mul(lev_B, axis=0)                          # la posicion decidida al cierre de t gana el rendimiento de t+1
pos_C = pos_A.mul(lev_C, axis=0)
pnls = {"A: carry base": pnl_A_full.iloc[CALENTAMIENTO:],
        "B: volatilidad objetivo": backtest(rend_tot, pos_B)[0].iloc[CALENTAMIENTO:],
        "C: media exposicion tras caida": backtest(rend_tot, pos_C)[0].iloc[CALENTAMIENTO:]}
tabla = pd.DataFrame({k: {**metricas(v), **cola(v)} for k, v in pnls.items()}).T
tabla["Calmar"] = tabla["rend_%"] / -tabla["caida_max_%"]
print(tabla.round(3).T.to_string())
print("\nApalancamiento B: promedio %.2f | minimo %.2f | maximo %.2f | dias en el tope %.0f%%" % (
    lev_B.iloc[CALENTAMIENTO:].mean(), lev_B.iloc[CALENTAMIENTO:].min(), lev_B.iloc[CALENTAMIENTO:].max(),
    (lev_B.iloc[CALENTAMIENTO:] >= TOPE - 1e-9).mean() * 100))
print("Exposicion C: dias con media exposicion %.0f%%" % ((lev_C.iloc[CALENTAMIENTO:] < 1).mean() * 100))
print("Error estandar aproximado del Sharpe: %.2f" % (1 / np.sqrt(len(pnls["A: carry base"]) / 252)))


                     A: carry base  B: volatilidad objetivo  C: media exposicion tras caida
rend_%                       4.372                    5.266                           3.743
vol_%                        4.669                    4.748                           4.218
Sharpe                       0.936                    1.109                           0.887
t                            1.925                    2.280                           1.824
p                            0.054                    0.023                           0.068
peor_mes_%                  -3.740                   -3.041                          -1.870
asimetria                   -0.768                   -0.270                          -0.204
caida_max_%                 -7.435                   -6.874                          -5.491
media_peor_5%_dia_%         -0.702                   -0.701                          -0.658
Calmar                       0.588                    0.766                     

In [5]:
# ¿LA VERSION B MEJORA DE VERDAD? Bootstrap por bloques de 21 dias (1000 remuestreos) de la diferencia de Sharpe contra A
rng = np.random.default_rng(5)
a, b, c = (pnls[k].values for k in pnls)
n, bloque = len(a), 21
dif_b, dif_c = [], []
for _ in range(1000):
    idx = np.concatenate([np.arange(s, s + bloque) % n for s in rng.integers(0, n, size=n // bloque + 1)])[:n]
    sa = a[idx].mean() / a[idx].std()
    dif_b.append((b[idx].mean() / b[idx].std() - sa) * np.sqrt(252))
    dif_c.append((c[idx].mean() / c[idx].std() - sa) * np.sqrt(252))
for nombre, d in (("B menos A", dif_b), ("C menos A", dif_c)):
    d = np.array(d)
    print("%s: diferencia de Sharpe promedio %.2f | intervalo de 95%%: %.2f a %.2f | fraccion mayor que cero: %.2f" % (
        nombre, d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d > 0).mean()))


B menos A: diferencia de Sharpe promedio 0.17 | intervalo de 95%: -0.04 a 0.35 | fraccion mayor que cero: 0.94
C menos A: diferencia de Sharpe promedio -0.05 | intervalo de 95%: -0.29 a 0.21 | fraccion mayor que cero: 0.34


In [6]:
# CONTROL: ¿importa CUANDO se reduce el riesgo? Se mezcla en el tiempo el apalancamiento de B (200 veces): mismo promedio, momentos al azar
rng = np.random.default_rng(11)
reales = {"Sharpe": sharpe(pnls["B: volatilidad objetivo"]), "caida_max_%": caida_maxima(pnls["B: volatilidad objetivo"]),
          "peor_mes_%": cola(pnls["B: volatilidad objetivo"])["peor_mes_%"]}
sims = {"Sharpe": [], "caida_max_%": [], "peor_mes_%": []}
for _ in range(200):
    lev_p = pd.Series(rng.permutation(lev_B.values), index=lev_B.index)
    p = backtest(rend_tot, pos_A.mul(lev_p, axis=0))[0].iloc[CALENTAMIENTO:]
    sims["Sharpe"].append(sharpe(p)); sims["caida_max_%"].append(caida_maxima(p)); sims["peor_mes_%"].append(cola(p)["peor_mes_%"])
for k in sims:
    s = np.array(sims[k])
    mejor = (s >= reales[k]).mean()          # para las tres medidas, mas alto es mejor (caida y peor mes son negativos)
    print("%-12s real %.2f | al azar: promedio %.2f, desv. est. %.2f | fraccion de azar igual o mejor que lo real: %.2f" % (
        k, reales[k], s.mean(), s.std(), mejor))


Sharpe       real 1.11 | al azar: promedio 0.92, desv. est. 0.09 | fraccion de azar igual o mejor que lo real: 0.01
caida_max_%  real -6.87 | al azar: promedio -8.12, desv. est. 0.70 | fraccion de azar igual o mejor que lo real: 0.04
peor_mes_%   real -3.04 | al azar: promedio -3.95, desv. est. 0.45 | fraccion de azar igual o mejor que lo real: 0.01


In [7]:
# Costos (puntos base por unidad operada: SUPUESTO mio). Cambiar el apalancamiento tambien cuesta.
filas = []
for nombre, pos in (("A: carry base", pos_A), ("B: volatilidad objetivo", pos_B), ("C: media exposicion tras caida", pos_C)):
    fila = {"estrategia": nombre, "rotacion_anual": backtest(rend_tot, pos, 1)[1].iloc[CALENTAMIENTO:].mean() * 252}
    for bps in (0, 1, 2, 5):
        fila[f"Sharpe {bps}bps"] = sharpe(backtest(rend_tot, pos, 1, bps)[0].iloc[CALENTAMIENTO:])
    filas.append(fila)
print(pd.DataFrame(filas).round(2).to_string(index=False))


                    estrategia  rotacion_anual  Sharpe 0bps  Sharpe 1bps  Sharpe 2bps  Sharpe 5bps
                 A: carry base           11.49         0.94         0.91         0.89         0.81
       B: volatilidad objetivo           19.94         1.11         1.07         1.03         0.90
C: media exposicion tras caida           13.92         0.89         0.85         0.82         0.72
